## __Phase 4: Evaluation (experiments)__

The full experiment battery (ranking metrics, the PHMM length correction with alpha
tuning, all four misclassification-bias experiments before/after correction, and the
reference-labeling comparison) now lives in
`scripts/evaluation/run_phase4_evaluation.py`, which saves every figure to
`reports/figures/phase4_evaluation/<error_model>/` and every table to
`reports/tables/phase4_evaluation/<error_model>/`:

```bash
python -m scripts.evaluation.run_phase4_evaluation --error-model uniform
```

This notebook keeps a light interactive version for exploration only: it imports that
script's own data-loading and metrics functions (nothing duplicated here) and plots a
couple of the headline figures live via `scripts.evaluation.plots`.

In [ ]:
import matplotlib.pyplot as plt

from scripts.evaluation import plots
from scripts.evaluation.metrics import summarize_ranking
from scripts.evaluation.run_phase4_evaluation import closed_set_metrics, load_data

In [ ]:
# Select which RF bearing-error model's data to evaluate (must match the
# --error-model used to generate/run the upstream scripts)
ERROR_MODEL = "uniform"  # "uniform" or "gaussian"
DATA_DIR = "../data/processed" if ERROR_MODEL == "uniform" else f"../data/processed/{ERROR_MODEL}"

data = load_data(DATA_DIR, base_dir="../data/processed")
data["preselection"].head(5)

### __Preselection overview__

In [ ]:
df_preselection_multimatch = data["preselection"].groupby(
    ["RF_track_id", "RF_signal_id"]).filter(
        lambda x: x["AIS_track_id"].count() > 1).reset_index(drop=True)

counts = data["preselection"].groupby(["RF_track_id", "RF_signal_id"]).size()
plots.candidate_count_histogram(counts)
plt.show()

### __Closed-set metrics (raw, uncorrected PHMM Forward score)__

In [ ]:
df_forward_results_prefilter, df_forward_results, df_with_chosen = closed_set_metrics(
    data, df_preselection_multimatch
)

### __Score margin: correct vs incorrect top-1 picks__

In [ ]:
ranking_summary_multi = summarize_ranking(df_forward_results, "normalized_forward_score", mode="max")
margin_data = ranking_summary_multi.dropna(subset=["margin"]).copy()
top1_correct = margin_data["rank_of_true"] == 1

plots.score_margin_histogram(
    margin_data.loc[top1_correct, "margin"],
    margin_data.loc[~top1_correct, "margin"],
    title="Top-1 score margin: correct vs incorrect picks (PHMM Forward)",
)
plt.show()